In [1]:
# ============================================================
# CELL 1 — IMPORTS + REPRODUCIBILITY + DEVICE
# ============================================================

import os
import gc
import sys
import json
import math
import time
import random
import zipfile
import importlib.util

from pathlib import Path
from collections import Counter

import numpy as np

# less GPU memory fragmentation (must be set before importing torch)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import precision_recall_fscore_support


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Device :", DEVICE)

if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Device : cuda
GPU    : Tesla T4


In [2]:
# ============================================================
# CELL 2 — CONFIGURATION
# Phase 3 (Rhetorical Roles) — KNOWLEDGE DISTILLATION (self-contained)
#
#   STEP 1  TEACHER: Architecture-1 decoder (frozen) + LoRMA + head  (~115M)
#           trained here from the Phase-1 checkpoint
#   STEP 2  teacher role scores for every train sentence
#   STEP 3  STUDENT: Architecture-2 encoder + head (~70M, all weights trained)
#           learns from the gold roles AND from the teacher's scores
#
# Only the Phase-1 checkpoints are needed (no Phase-3 outputs).
# Both models are trained on TRAIN only; dev only picks the best epoch.
# ============================================================

# 0.5 = KD run. 0.0 = the same student WITHOUT the teacher (baseline for the report)
KD_ALPHA = 0.5               # loss = (1 - KD_ALPHA) * gold loss + KD_ALPHA * distillation loss
KD_TEMPERATURE = 2.0         # softens the teacher's probabilities
KD_ONLY_WHERE_TEACHER_CORRECT = True   # distil only sentences the teacher labels correctly

RUN_NAME = "p3_kd_arch1lorma_to_arch2" if KD_ALPHA > 0 else "p3_arch2_no_kd"


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

TOKENIZER_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/"
    "legal_tokenizer_v3/tokenizer/final_tokenizer"
)

# Phase-1 Architecture-1 checkpoint (the teacher's frozen decoder).
# If a checkpoint path is wrong, CELL 11 / CELL 13 search /kaggle/input for it.
TEACHER_CHECKPOINT = Path(
    "/kaggle/input/models/ritikguptatab/proj-cap-p-1-archi-1/"
    "pytorch/default/1/best_model.pt"
)

# Phase-1 Architecture-2 checkpoint (the student's encoder)
STUDENT_CHECKPOINT = Path(
    "/kaggle/input/models/ritikguptatab/proj-cap-p-1-archi-2/"
    "pytorch/default/1/best_encoder_decoder.pt"
)

# Phase-3 data: None = download from Hugging Face (Kaggle secret "HF"),
# or the folder that contains Train/ and Test/, e.g. Path("/kaggle/input/phase-3-rr/Phase_3_RR")
LOCAL_DATA_DIR = None
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

OUTPUT_DIR = Path(f"/kaggle/working/{RUN_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Sentences and windows (the same for teacher and student)
# ------------------------------------------------------------

MAX_SENT_TOKENS = 256        # longer sentences are cut
WINDOW = 1024                # consecutive sentences are packed into windows of 1024 tokens

TEACHER_WINDOWS_PER_CHUNK = 8    # windows per teacher training step. If CUDA out-of-memory: 4
STUDENT_WINDOWS_PER_CHUNK = 4    # windows per student training step (all weights train). If out-of-memory: 2


# ------------------------------------------------------------
# Teacher: Architecture-1 decoder (FROZEN) + LoRMA
#   W' = (I + alpha/r * B A) W,  only A and B are trained
# ------------------------------------------------------------

LORMA_TARGETS = ["qkv", "out_proj", "gate", "up", "down"]    # every linear layer of every block
LORMA_RANK = 16
LORMA_ALPHA = 32
LORMA_DROPOUT = 0.05

LR_LORMA = 5e-4
TEACHER_EPOCHS = 15


# ------------------------------------------------------------
# Head (same design for teacher and student, each has its own)
#   sentence vectors -> LayerNorm -> BiLSTM over the document -> CRF
# ------------------------------------------------------------

LSTM_HIDDEN = 256
HEAD_DROPOUT = 0.3

CLASS_BALANCING = True       # rare roles get a larger loss weight (macro-F1 counts every role equally)
CLASS_WEIGHT_MAX = 5.0       # cap, so a role with 3 sentences does not dominate
CE_WEIGHT = 0.5              # gold loss = CRF + CE_WEIGHT * class-weighted cross-entropy

# A role with fewer than MIN_LABEL_COUNT train sentences in a domain is never
# predicted in that domain (e.g. Dissent in IT: 0 sentences).
MIN_LABEL_COUNT = 10


# ------------------------------------------------------------
# Student: Architecture-2 encoder, FULL fine-tuning
# ------------------------------------------------------------

LR_STUDENT_ENCODER = 1e-4    # same as the Phase-2 student
WEIGHT_DECAY = 0.01          # student encoder only
STUDENT_EPOCHS = 15


# ------------------------------------------------------------
# Training (both models)
# ------------------------------------------------------------

PATIENCE = 5                 # stop if the dev score does not improve for 5 epochs
HEAD_ONLY_EPOCHS = 1         # epoch 1 trains only the head (backbone unchanged)
LR_HEAD = 2e-3
WARMUP_RATIO = 0.06
GRAD_CLIP = 1.0
TRAIN_EVAL_DOCS = 10         # a few train documents are scored every epoch (to see under-fitting)


# ------------------------------------------------------------
# Debug mode (quick run on a few documents)
# ------------------------------------------------------------

DEBUG_MODE = False
DEBUG_DOCS = 6               # per file

if DEBUG_MODE:
    TEACHER_EPOCHS = STUDENT_EPOCHS = 2


# ------------------------------------------------------------
# Mixed precision (T4 / P100 -> float16 + GradScaler, A100 -> bfloat16)
# ------------------------------------------------------------

USE_AMP = DEVICE.type == "cuda"

if USE_AMP and torch.cuda.get_device_capability()[0] >= 8:
    AMP_DTYPE = torch.bfloat16
else:
    AMP_DTYPE = torch.float16

USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16


def autocast():
    return torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)


assert 0.0 <= KD_ALPHA < 1.0
assert MAX_SENT_TOKENS <= WINDOW

print("Run       :", RUN_NAME, f"(KD alpha {KD_ALPHA}, temperature {KD_TEMPERATURE})")
print("Output    :", OUTPUT_DIR)
print("AMP dtype :", AMP_DTYPE if USE_AMP else "disabled")

Run       : p3_kd_arch1lorma_to_arch2 (KD alpha 0.5, temperature 2.0)
Output    : /kaggle/working/p3_kd_arch1lorma_to_arch2
AMP dtype : torch.float16


In [3]:
# ============================================================
# CELL 3 — LOAD THE PHASE-1 TOKENIZER
# ============================================================

spec = importlib.util.spec_from_file_location("legal_tokenizer_module", TOKENIZER_DIR / "tokenizer.py")
tok_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tok_module)

tokenizer, tokenizer_config = tok_module.load_tokenizer(TOKENIZER_DIR)

PAD_ID = tokenizer.pad_id
UNK_ID = tokenizer.unk_id
VOCAB_SIZE = tokenizer.get_vocab_size()

print("Vocabulary size:", VOCAB_SIZE, "| PAD / UNK:", PAD_ID, UNK_ID)


token_cache = {}      # sentence -> token ids (legal documents repeat many sentences)


def encode_sentence(sentence):
    """
    Token ids of one sentence (NOT cut yet; CELL 6 cuts to MAX_SENT_TOKENS).
    A space is added in front: inside a document every sentence follows a
    space, and the tokenizer keeps the space in front of the first word.
    """

    if sentence not in token_cache:

        ids = tokenizer.encode(" " + sentence.strip())

        token_cache[sentence] = ids if ids else [UNK_ID]   # empty sentence -> 1 token, so it still gets a vector

    return token_cache[sentence]


demo = "The petitioner filed the present appeal."
print("\nDemo:", [tokenizer.id_to_token[i] for i in encode_sentence(demo)])

Vocabulary size: 32004 | PAD / UNK: 0 1

Demo: [' ', 'The', ' ', 'petitioner', ' ', 'filed', ' ', 'the', ' ', 'present', ' ', 'appeal', '.']


In [4]:
# ============================================================
# CELL 4 — PHASE-3 DATA FILES  (download needs the Kaggle secret "HF")
# ============================================================

PHASE3_FILES = {
    "CL_train": "Phase_3_RR/Train/CL_train.json",
    "CL_dev":   "Phase_3_RR/Train/CL_dev.json",
    "IT_train": "Phase_3_RR/Train/IT_train.json",
    "IT_dev":   "Phase_3_RR/Train/IT_dev.json",
    "labels":   "Phase_3_RR/Train/label_vocab.json",
    "CL_test":  "Phase_3_RR/Test/CL_test.json",
    "IT_test":  "Phase_3_RR/Test/IT_test.json",
}

DATA_FILES = {}

if LOCAL_DATA_DIR is not None:

    for name, filename in PHASE3_FILES.items():
        DATA_FILES[name] = Path(LOCAL_DATA_DIR) / filename.replace("Phase_3_RR/", "")

else:

    from huggingface_hub import hf_hub_download
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF")

    for name, filename in PHASE3_FILES.items():

        DATA_FILES[name] = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename=filename,
            repo_type="dataset",
            token=HF_TOKEN,
        )

for name, path in DATA_FILES.items():
    print(f"{name:9s} -> {path}")

CL_train  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Train/CL_train.json
CL_dev    -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Train/CL_dev.json
IT_train  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Train/IT_train.json
IT_dev    -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Train/IT_dev.json
labels    -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Train/label_vocab.json
CL_test   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Test/CL_test.json
I

In [5]:
# ============================================================
# CELL 5 — LOAD DOCUMENTS + LABELS + STATISTICS
#
# Every document: {"id", "domain", "sentences": [...], "labels": [...] or None}
# ============================================================

def read_json(path):
    """A JSON file, or a JSON-lines file (one document per line)."""

    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return [json.loads(line) for line in content.splitlines() if line.strip()]


def load_label_vocab(path):

    raw = read_json(path)

    if isinstance(raw, dict) and isinstance(raw.get("labels"), list):
        raw = raw["labels"]

    if isinstance(raw, list):
        return [str(label) for label in raw]

    # dict: {label: id} or {id: label}
    if all(isinstance(v, int) for v in raw.values()):
        return [label for label, _ in sorted(raw.items(), key=lambda item: item[1])]

    return [str(raw[key]) for key in sorted(raw, key=int)]


def load_documents(path, domain):

    raw = read_json(path)

    # accepted layouts: [doc, ...]  |  {"data": [doc, ...]}  |  {doc_id: doc}
    if isinstance(raw, dict):

        list_key = next((k for k in ("data", "documents", "docs") if isinstance(raw.get(k), list)), None)

        if list_key is not None:
            raw = raw[list_key]
        else:
            raw = [
                dict(value, id=key) if isinstance(value, dict) else {"id": key, "text": value}
                for key, value in raw.items()
            ]

    docs = []

    for i, item in enumerate(raw):

        sentences = item.get("text", item.get("sentences"))
        labels = item.get("labels", item.get("label"))

        doc = {
            "id": str(item.get("id", f"{domain}_{i}")),
            "domain": domain,
            "sentences": [str(s) for s in sentences],
            "labels": None if labels is None else [str(label) for label in labels],
        }

        if doc["labels"] is not None:
            assert len(doc["labels"]) == len(doc["sentences"]), f"{doc['id']}: labels and sentences differ in length"

        docs.append(doc)

    if DEBUG_MODE:
        docs = docs[:DEBUG_DOCS]

    return docs


LABELS = load_label_vocab(DATA_FILES["labels"])
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
NUM_LABELS = len(LABELS)

print(f"{NUM_LABELS} labels:", LABELS)

train_docs = load_documents(DATA_FILES["CL_train"], "CL") + load_documents(DATA_FILES["IT_train"], "IT")
dev_docs = load_documents(DATA_FILES["CL_dev"], "CL") + load_documents(DATA_FILES["IT_dev"], "IT")
test_docs = load_documents(DATA_FILES["CL_test"], "CL") + load_documents(DATA_FILES["IT_test"], "IT")


# ------------------------------------------------------------
# Label ids. A label that is not in label_vocab.json is reported
# and mapped to "None".
# ------------------------------------------------------------

unknown = Counter()

for doc in train_docs + dev_docs:

    for k, label in enumerate(doc["labels"]):
        if label not in LABEL2ID:
            unknown[label] += 1
            doc["labels"][k] = "None"

    doc["label_ids"] = [LABEL2ID[label] for label in doc["labels"]]

if unknown:
    print("\nWARNING: labels not in label_vocab.json (mapped to 'None'):", dict(unknown))


# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print(f"\n{'split':6s} {'domain':6s} {'docs':>5s} {'sentences':>10s} {'per doc (mean / max)':>22s}")

for split, docs in [("train", train_docs), ("dev", dev_docs), ("test", test_docs)]:
    for domain in ["CL", "IT"]:

        lengths = [len(d["sentences"]) for d in docs if d["domain"] == domain]

        if lengths:
            print(f"{split:6s} {domain:6s} {len(lengths):5d} {sum(lengths):10d} "
                  f"{np.mean(lengths):14.1f} / {max(lengths):5d}")

print(f"\nLabel distribution (train | dev):")
print(f"{'label':24s} {'CL train':>9s} {'IT train':>9s} {'CL dev':>7s} {'IT dev':>7s}")

counts = {
    (split, domain): Counter(label for d in docs if d["domain"] == domain for label in d["labels"])
    for split, docs in [("train", train_docs), ("dev", dev_docs)]
    for domain in ["CL", "IT"]
}

for label in LABELS:
    print(f"{label:24s} {counts['train', 'CL'][label]:9d} {counts['train', 'IT'][label]:9d} "
          f"{counts['dev', 'CL'][label]:7d} {counts['dev', 'IT'][label]:7d}")


# ------------------------------------------------------------
# Class weights for the loss: (N / (C * n_label)) ** 0.5, at most CLASS_WEIGHT_MAX.
# The square root keeps the weights moderate.
# ------------------------------------------------------------

train_counts = Counter(label for d in train_docs for label in d["labels"])
n_train = sum(train_counts.values())

if CLASS_BALANCING:
    CLASS_WEIGHTS = torch.tensor(
        [min(CLASS_WEIGHT_MAX, (n_train / (NUM_LABELS * max(train_counts[label], 1))) ** 0.5) for label in LABELS]
    )
else:
    CLASS_WEIGHTS = torch.ones(NUM_LABELS)

print("\nClass weights:", {label: round(w, 2) for label, w in zip(LABELS, CLASS_WEIGHTS.tolist())})


# ------------------------------------------------------------
# Roles allowed per domain (used at prediction time, CELL 11)
# ------------------------------------------------------------

LABEL_ALLOWED = {
    domain: np.array([counts["train", domain][label] >= MIN_LABEL_COUNT for label in LABELS])
    for domain in ["CL", "IT"]
}

for domain in ["CL", "IT"]:
    never = [label for label, ok in zip(LABELS, LABEL_ALLOWED[domain]) if not ok]
    print(f"Never predicted in {domain} (< {MIN_LABEL_COUNT} train sentences):", never)
print("\nSample:", train_docs[0]["id"], "|", train_docs[0]["labels"][0], "|", train_docs[0]["sentences"][0][:120])


split  domain  docs  sentences   per doc (mean / max)
train  CL        40      10593          264.8 /  1816
train  IT        40       6252          156.3 /   442
dev    CL         5       1396          279.2 /   848
dev    IT         5        746          149.2 /   382
test   CL         5       1339          267.8 /   942
test   IT         5        858          171.6 /   420

Label distribution (train | dev):
label                     CL train  IT train  CL dev  IT dev
Fact                          2979      1119     249     195
Issue                           17        90       3      12
ArgumentPetitioner             976       637     344      56
ArgumentRespondent             996       352     236      36
Statute                        367       732      58      62
Dissent                        284         0       0       0
PrecedentReliedUpon           1305       640     142     112
PrecedentNotReliedUpon         209       360       0      50
PrecedentOverruled               0   

In [6]:
# ============================================================
# CELL 6 — TOKENIZE + WINDOWS + CHUNKS
#
# window = token ids that go through the model together,
#          {"ids": [...], "sentences": [(sentence_index, start, end), ...]}
#   consecutive sentences are packed into one window (up to WINDOW
#   tokens), so every sentence also sees its neighbours.
#   Teacher and student read the SAME windows.
#
# chunk = the windows of ONE training step (consecutive sentences
#         of one document). The teacher uses bigger chunks (only LoRMA
#         and the head train, so it needs less memory).
# ============================================================

def pack_windows(doc):

    windows = []

    for i, sentence in enumerate(doc["sentences"]):

        ids = encode_sentence(sentence)[:MAX_SENT_TOKENS]

        if not windows or len(windows[-1]["ids"]) + len(ids) > WINDOW:
            windows.append({"ids": [], "sentences": []})

        window = windows[-1]

        window["sentences"].append((i, len(window["ids"]), len(window["ids"]) + len(ids)))
        window["ids"] = window["ids"] + ids

    return windows


def make_chunks(docs, windows_per_chunk):

    chunks = []

    for doc in docs:
        for k in range(0, len(doc["windows"]), windows_per_chunk):
            chunks.append({"doc": doc, "windows": doc["windows"][k:k + windows_per_chunk]})

    return chunks


t0 = time.time()

for doc in train_docs + dev_docs + test_docs:

    doc["windows"] = pack_windows(doc)

    # every sentence must be in exactly one window, in order
    order = [i for window in doc["windows"] for i, _, _ in window["sentences"]]
    assert order == list(range(len(doc["sentences"]))), doc["id"]

print(f"Tokenized in {time.time() - t0:.1f}s")

lengths = np.array([len(encode_sentence(s)) for d in train_docs for s in d["sentences"]])

print(f"Sentences longer than {MAX_SENT_TOKENS} tokens (cut): {(lengths > MAX_SENT_TOKENS).mean():.2%}")
print(f"Train windows: {sum(len(d['windows']) for d in train_docs)} | steps per epoch: "
      f"teacher {len(make_chunks(train_docs, TEACHER_WINDOWS_PER_CHUNK))}, "
      f"student {len(make_chunks(train_docs, STUDENT_WINDOWS_PER_CHUNK))}")

Tokenized in 5.9s
Sentences longer than 256 tokens (cut): 1.92%
Train windows: 1435 | steps per epoch: teacher 212, student 388


In [7]:
# ============================================================
# CELL 7 — SCORES + VITERBI
# ============================================================

def macro_scores(gold, pred):
    """Macro precision / recall / F1 over the labels that occur in gold or prediction."""

    labels = sorted(set(gold) | set(pred))

    p, r, f, _ = precision_recall_fscore_support(gold, pred, labels=labels, average="macro", zero_division=0)

    return {"precision": float(p), "recall": float(r), "f1": float(f)}


def flat_labels(docs, predictions, domain=None):
    """Gold and predicted labels of all sentences, in order (one domain or all)."""

    gold, pred = [], []

    for doc in docs:
        if domain is None or doc["domain"] == domain:
            gold.extend(doc["labels"])
            pred.extend(predictions[doc["id"]])

    return gold, pred


def domain_scores(docs, predictions):
    """
    Macro scores for CL, IT and all sentences together.
    "score" = mean of the CL and IT macro-F1 (both domains count equally);
    it is used to pick the best epoch.
    """

    scores = {}

    for domain in ["CL", "IT"]:

        gold, pred = flat_labels(docs, predictions, domain)

        if gold:
            scores[domain] = macro_scores(gold, pred)

    scores["all"] = macro_scores(*flat_labels(docs, predictions))
    scores["score"] = float(np.mean([scores[d]["f1"] for d in ["CL", "IT"] if d in scores]))

    return scores


def viterbi(emissions, transitions, start_scores, end_scores):
    """emissions [n_sentences, NUM_LABELS] -> best label id per sentence (numpy)."""

    n = len(emissions)

    score = start_scores + emissions[0]
    backpointer = np.zeros((n, NUM_LABELS), dtype=np.int64)

    for i in range(1, n):
        candidates = score[:, None] + transitions          # [previous, current]
        backpointer[i] = candidates.argmax(axis=0)
        score = candidates.max(axis=0) + emissions[i]

    path = [int((score + end_scores).argmax())]

    for i in range(n - 1, 0, -1):
        path.append(int(backpointer[i, path[-1]]))

    return np.array(path[::-1], dtype=np.int64)


_g = ["Fact", "Fact", "Issue", "Statute"]
_p = ["Fact", "Issue", "Issue", "Statute"]
print("macro_scores check:", macro_scores(_g, _p))

macro_scores check: {'precision': 0.8333333333333334, 'recall': 0.8333333333333334, 'f1': 0.7777777777777777}


In [8]:
# ============================================================
# CELL 8 — MODELS
#
#   TEACHER backbone: Architecture-1 DECODER (frozen) + LoRMA
#   STUDENT backbone: Architecture-2 ENCODER (all weights trained)
#
#   head (same for both):
#     mean of the token vectors of each sentence = sentence vector
#     -> LayerNorm, + relative position + domain flag
#     -> BiLSTM over the sentences of the document -> CRF
#
# Layer names match the Phase-1 notebooks, so the weights load directly.
# ============================================================

# ============================================================
# Architecture-1 DECODER (teacher backbone)
# ============================================================

class RMSNorm(nn.Module):

    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x_float = x.float()
        rms = torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x_float * rms * self.weight).to(x.dtype)


def build_rope_cache(block_size, head_dim, theta):

    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    angles = torch.outer(torch.arange(block_size).float(), inv_freq)

    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):

    x1, x2 = x.float().chunk(2, dim=-1)

    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).to(x.dtype)


class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()

        d_model = config["d_model"]

        self.n_heads = config["n_heads"]
        self.head_dim = d_model // self.n_heads

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.q_norm = RMSNorm(self.head_dim)
        self.k_norm = RMSNorm(self.head_dim)

    def forward(self, x, cos, sin):

        B, T, C = x.shape

        q, k, v = self.qkv(x).split(C, dim=-1)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        q = apply_rope(self.q_norm(q), cos, sin)
        k = apply_rope(self.k_norm(k), cos, sin)

        # causal, as in pretraining. Padding is at the END, so real tokens never see it.
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        return self.out_proj(y.transpose(1, 2).contiguous().view(B, T, C))


class SwiGLU(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.gate = nn.Linear(config["d_model"], config["d_ff"], bias=False)
        self.up = nn.Linear(config["d_model"], config["d_ff"], bias=False)
        self.down = nn.Linear(config["d_ff"], config["d_model"], bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class TransformerBlock(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.norm1 = RMSNorm(config["d_model"])
        self.attn = CausalSelfAttention(config)
        self.norm2 = RMSNorm(config["d_model"])
        self.mlp = SwiGLU(config)

    def forward(self, x, cos, sin):

        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.mlp(self.norm2(x))

        return x


class DecoderBackbone(nn.Module):
    """Architecture-1 without the LM head: tokens -> hidden vectors."""

    def __init__(self, config):
        super().__init__()

        self.token_embedding = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config["n_layers"])])
        self.norm_f = RMSNorm(config["d_model"])

        cos, sin = build_rope_cache(config["block_size"], config["d_model"] // config["n_heads"], config["rope_theta"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, input_ids, mask=None):
        """`mask` is not needed: attention is causal and padding is at the end."""

        T = input_ids.size(1)

        cos, sin = self.rope_cos[:T], self.rope_sin[:T]

        x = self.token_embedding(input_ids)

        for block in self.blocks:
            x = block(x, cos, sin)

        return self.norm_f(x)


# ------------------------------------------------------------
# LoRMA (multiplicative low-rank adaptation)
#     W' x = (I + alpha/r * B A) W x = h + alpha/r * B (A h),   h = W x
# A: out_features -> r,  B: r -> out_features,  B = 0 at the start (W' = W).
# ------------------------------------------------------------

class LoRMALinear(nn.Module):

    def __init__(self, base, rank, alpha, dropout):
        super().__init__()

        self.base = base                                      # frozen Phase-1 layer
        out_features = base.out_features

        self.lorma_A = nn.Linear(out_features, rank, bias=False)
        self.lorma_B = nn.Linear(rank, out_features, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale = alpha / rank

        nn.init.kaiming_uniform_(self.lorma_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_B.weight)                   # starts as the identity -> W' = W

    def forward(self, x):

        h = self.base(x)

        return h + self.scale * self.lorma_B(self.lorma_A(self.dropout(h)))


def add_lorma(backbone):
    """Freeze every Phase-1 weight, then wrap the target linear layers with LoRMA."""

    for p in backbone.parameters():
        p.requires_grad = False

    for block in backbone.blocks:
        for name in LORMA_TARGETS:

            parent = block.attn if name in ("qkv", "out_proj") else block.mlp
            setattr(parent, name, LoRMALinear(getattr(parent, name), LORMA_RANK, LORMA_ALPHA, LORMA_DROPOUT))


# ============================================================
# Architecture-2 ENCODER (student backbone)
# ============================================================

class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, n_heads, dropout):
        super().__init__()

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        B, T, D = x.shape
        return x.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, x, attn_mask):

        q, k, v = self.split_heads(self.q_proj(x)), self.split_heads(self.k_proj(x)), self.split_heads(self.v_proj(x))

        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                             dropout_p=self.dropout if self.training else 0.0)
        B, H, T, d = out.shape

        return self.out_proj(out.transpose(1, 2).reshape(B, T, H * d))


class PositionalEmbedding(nn.Module):

    def __init__(self, max_len, d_model, dropout):
        super().__init__()
        self.embedding = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device)
        return self.dropout(x + self.embedding(positions).unsqueeze(0))


class FeedForward(nn.Module):

    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.fc2(F.gelu(self.fc1(x))))


class EncoderBlock(nn.Module):

    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, attn_mask):
        x = x + self.self_attention(self.norm1(x), attn_mask)
        return x + self.feed_forward(self.norm2(x))


class Arch2Encoder(nn.Module):
    """Architecture-2 encoder: bidirectional, learned positions."""

    def __init__(self, cfg):
        super().__init__()

        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.position_embedding = PositionalEmbedding(cfg["src_len"], cfg["d_model"], cfg["dropout"])
        self.layers = nn.ModuleList([
            EncoderBlock(cfg["d_model"], cfg["n_heads"], cfg["ffn_dim"], cfg["dropout"])
            for _ in range(cfg["encoder_layers"])
        ])
        self.final_norm = nn.LayerNorm(cfg["d_model"])

    def forward(self, input_ids, mask):

        attn_mask = mask[:, None, None, :]                    # padding is never attended to
        x = self.position_embedding(self.token_embedding(input_ids))

        for layer in self.layers:
            x = layer(x, attn_mask)

        return self.final_norm(x)


# ============================================================
# CRF + sentence-role model (same head for teacher and student)
# ============================================================

class CRF(nn.Module):
    """Linear-chain CRF over the sentences of a document (learns which role follows which)."""

    def __init__(self, num_labels):
        super().__init__()

        self.transitions = nn.Parameter(torch.zeros(num_labels, num_labels))      # [previous, current]
        self.start = nn.Parameter(torch.zeros(num_labels))
        self.end = nn.Parameter(torch.zeros(num_labels))

    def loss(self, emissions, labels):
        """Negative log-likelihood per sentence. emissions [S, C] (float32), labels [S]."""

        S = emissions.size(0)

        gold = (
            self.start[labels[0]]
            + emissions[torch.arange(S, device=emissions.device), labels].sum()
            + self.transitions[labels[:-1], labels[1:]].sum()
            + self.end[labels[-1]]
        )

        alpha = self.start + emissions[0]

        for i in range(1, S):
            alpha = torch.logsumexp(alpha.unsqueeze(1) + self.transitions, dim=0) + emissions[i]

        log_z = torch.logsumexp(alpha + self.end, dim=0)

        return (log_z - gold) / S


class SentenceRoleModel(nn.Module):
    """backbone (teacher or student) + head."""

    def __init__(self, backbone, d_model):
        super().__init__()

        self.backbone = backbone
        self.dropout = nn.Dropout(HEAD_DROPOUT)

        # LayerNorm: sentence vectors of any size -> mean 0, std 1 (keeps the BiLSTM from saturating)
        self.vector_norm = nn.LayerNorm(d_model)

        self.lstm = nn.LSTM(d_model + 2, LSTM_HIDDEN, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(2 * LSTM_HIDDEN, NUM_LABELS)
        self.crf = CRF(NUM_LABELS)

        self.register_buffer("class_weights", CLASS_WEIGHTS.clone())

    def head_parameters(self):
        return [p for name, p in self.named_parameters() if not name.startswith("backbone.")]

    def sentence_vectors(self, windows):
        """windows -> (vectors [S, D] float32, sentence_index [S]). Sentence vector = mean of its token vectors."""

        max_len = max(len(w["ids"]) for w in windows)
        ids = torch.full((len(windows), max_len), PAD_ID, dtype=torch.long)
        mask = torch.zeros((len(windows), max_len), dtype=torch.bool)

        for row, window in enumerate(windows):
            ids[row, :len(window["ids"])] = torch.tensor(window["ids"])
            mask[row, :len(window["ids"])] = True

        with autocast():
            hidden = self.backbone(ids.to(DEVICE), mask.to(DEVICE))           # [W, T, D]

        vectors = []
        sentence_index = []

        for row, window in enumerate(windows):
            for i, start, end in window["sentences"]:
                vectors.append(hidden[row, start:end].float().mean(dim=0))
                sentence_index.append(i)

        return torch.stack(vectors), sentence_index

    def head(self, vectors, doc, sentence_index):
        """Sentence vectors [S, D] -> label scores [S, NUM_LABELS] (float32)."""

        n = len(doc["sentences"])

        position = torch.tensor([[i / max(n - 1, 1)] for i in sentence_index], device=DEVICE)
        domain = torch.full_like(position, 1.0 if doc["domain"] == "IT" else 0.0)

        x = self.dropout(torch.cat([self.vector_norm(vectors), position, domain], dim=-1))

        out, _ = self.lstm(x.unsqueeze(0))              # one sequence = the sentences in document order

        return self.classifier(self.dropout(out.squeeze(0)))

    def chunk_emissions(self, chunk):
        """Label scores of the sentences of one training chunk."""

        vectors, sentence_index = self.sentence_vectors(chunk["windows"])

        return self.head(vectors, chunk["doc"], sentence_index), sentence_index


# ------------------------------------------------------------
# Check: LoRMA really computes (I + alpha/r * B A) W x
# ------------------------------------------------------------

_layer = LoRMALinear(nn.Linear(8, 6, bias=False), rank=2, alpha=4, dropout=0.0)
nn.init.normal_(_layer.lorma_B.weight)
_x = torch.randn(3, 8)

_merged = (torch.eye(6) + _layer.scale * _layer.lorma_B.weight @ _layer.lorma_A.weight) @ _layer.base.weight

assert torch.allclose(_layer(_x), _x @ _merged.T, atol=1e-5)
print("✅ models defined | LoRMA layer = (I + alpha/r * B A) W")

✅ models defined | LoRMA layer = (I + alpha/r * B A) W


In [9]:
# ============================================================
# CELL 9 — PREDICTION + EVALUATION (any model: teacher or student)
#
# A whole document goes through the model (windows in batches),
# then the head sees all its sentences at once and the CRF Viterbi
# picks the best role sequence. Roles not allowed in the document's
# domain (CELL 5) are never picked. One role per sentence.
# ============================================================

EVAL_WINDOWS = 8          # windows per forward pass


@torch.no_grad()
def document_emissions(model, doc):
    """Label scores of every sentence of one document: numpy [n_sentences, NUM_LABELS].
    The model must be in eval mode."""

    vectors, sentence_index = [], []

    for k in range(0, len(doc["windows"]), EVAL_WINDOWS):

        v, idx = model.sentence_vectors(doc["windows"][k:k + EVAL_WINDOWS])

        vectors.append(v)
        sentence_index.extend(idx)

    return model.head(torch.cat(vectors), doc, sentence_index).float().cpu().numpy()


@torch.no_grad()
def predict_labels(model, docs):
    """{doc_id: [label, label, ...]}  one label per sentence, in order."""

    model.eval()

    transitions = model.crf.transitions.float().cpu().numpy()
    start = model.crf.start.float().cpu().numpy()
    end = model.crf.end.float().cpu().numpy()

    predictions = {}

    for doc in docs:

        if not doc["sentences"]:
            predictions[doc["id"]] = []
            continue

        emissions = document_emissions(model, doc)
        emissions[:, ~LABEL_ALLOWED[doc["domain"]]] = -1e4

        predictions[doc["id"]] = [LABELS[i] for i in viterbi(emissions, transitions, start, end)]

    model.train()

    return predictions


def evaluate(model, docs):

    predictions = predict_labels(model, docs)

    return domain_scores(docs, predictions), predictions

In [10]:
# ============================================================
# CELL 10 — LOSSES + OPTIMIZER + TRAINING FUNCTION (used for teacher AND student)
#
#   gold loss = CRF + CE_WEIGHT * class-weighted cross-entropy
#   KD loss   = T^2 * KL( teacher probs || student probs ), both at temperature T
#               (Hinton et al., 2015), only on sentences the teacher gets right
#   teacher loss = gold loss
#   student loss = (1 - KD_ALPHA) * gold loss + KD_ALPHA * KD loss
# ============================================================

def gold_loss(model, chunk):
    """Returns (gold loss, emissions, labels, sentence_index) for one chunk."""

    doc = chunk["doc"]

    emissions, sentence_index = model.chunk_emissions(chunk)
    labels = torch.tensor([doc["label_ids"][i] for i in sentence_index], device=DEVICE)

    loss = model.crf.loss(emissions, labels) + CE_WEIGHT * F.cross_entropy(emissions, labels, weight=model.class_weights)

    return loss, emissions, labels, sentence_index


def distillation_loss(student_scores, teacher_scores, mask):

    T = KD_TEMPERATURE

    student_log_probs = F.log_softmax(student_scores / T, dim=-1)
    teacher_probs = F.softmax(teacher_scores / T, dim=-1)

    kl = (teacher_probs * (torch.log(teacher_probs + 1e-9) - student_log_probs)).sum(dim=-1)     # [S]

    return (kl * mask).sum() / mask.sum().clamp(min=1.0) * T * T


def teacher_loss(model, chunk):
    """Returns (total loss, gold loss, KD loss)."""

    gold, _, _, _ = gold_loss(model, chunk)

    return gold, gold, torch.zeros((), device=DEVICE)


def student_loss(model, chunk):
    """Returns (total loss, gold loss, KD loss)."""

    gold, emissions, labels, sentence_index = gold_loss(model, chunk)

    if KD_ALPHA == 0:
        return gold, gold, torch.zeros((), device=DEVICE)

    teacher_scores = chunk["doc"]["teacher_scores"][sentence_index].to(DEVICE)
    mask = torch.ones(len(sentence_index), device=DEVICE)

    if KD_ONLY_WHERE_TEACHER_CORRECT:
        mask = (teacher_scores.argmax(dim=-1) == labels).float()

    kd = distillation_loss(emissions, teacher_scores, mask)

    return (1 - KD_ALPHA) * gold + KD_ALPHA * kd, gold, kd


def make_optimizer(model, total_steps, lr_backbone, weight_decay_backbone):
    """group 0 = trainable backbone weights (teacher: LoRMA, student: whole encoder), group 1 = head."""

    optimizer = torch.optim.AdamW(
        [
            {"params": [p for p in model.backbone.parameters() if p.requires_grad],
             "lr": lr_backbone, "weight_decay": weight_decay_backbone},
            {"params": model.head_parameters(), "lr": LR_HEAD, "weight_decay": 0.0},
        ],
        betas=(0.9, 0.999),
    )

    base_lrs = [group["lr"] for group in optimizer.param_groups]
    warmup_steps = max(1, int(WARMUP_RATIO * total_steps))

    def set_lr(step):
        """Linear warmup, then linear decay to 0."""

        if step < warmup_steps:
            factor = (step + 1) / warmup_steps
        else:
            factor = max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))

        for group, base_lr in zip(optimizer.param_groups, base_lrs):
            group["lr"] = base_lr * factor

    return optimizer, set_lr


def trainable_state(model):
    """Only the trained weights (teacher: LoRMA + head, student: everything)."""

    return {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}


def train_model(model, loss_fn, epochs, windows_per_chunk, lr_backbone, weight_decay_backbone):
    """
    Trains `model` on TRAIN documents only. After every epoch it is scored on dev;
    the best epoch is kept, and training stops after PATIENCE epochs without improvement.
    Returns (history, best_epoch).
    """

    chunks = make_chunks(train_docs, windows_per_chunk)

    optimizer, set_lr = make_optimizer(model, epochs * len(chunks), lr_backbone, weight_decay_backbone)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)

    train_check_docs = random.Random(SEED).sample(train_docs, min(TRAIN_EVAL_DOCS, len(train_docs)))

    history = []
    best_score, best_epoch, best_state = -1.0, 0, None
    step = 0
    t_start = time.time()

    model.train()

    for epoch in range(1, epochs + 1):

        random.shuffle(chunks)
        sum_loss = sum_gold = sum_kd = 0.0
        skipped = 0                   # steps skipped by the float16 GradScaler (overflow)
        head_only = epoch <= HEAD_ONLY_EPOCHS

        for chunk in chunks:

            set_lr(step)

            if head_only:
                optimizer.param_groups[0]["lr"] = 0.0          # group 0 = backbone: not trained yet

            loss, gold, kd = loss_fn(model, chunk)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            # clip backbone and head separately
            torch.nn.utils.clip_grad_norm_(optimizer.param_groups[0]["params"], GRAD_CLIP)
            torch.nn.utils.clip_grad_norm_(optimizer.param_groups[1]["params"], GRAD_CLIP)

            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            skipped += scaler.get_scale() < scale_before

            step += 1
            sum_loss += loss.item()
            sum_gold += gold.item()
            sum_kd += kd.item()

        n = len(chunks)
        train_scores, _ = evaluate(model, train_check_docs)
        dev_scores, _ = evaluate(model, dev_docs)

        log = {"epoch": epoch, "loss": sum_loss / n, "gold_loss": sum_gold / n, "kd_loss": sum_kd / n,
               "train_f1": train_scores["score"], "dev": dev_scores, "head_only": head_only,
               "skipped_steps": int(skipped)}

        line = f"Epoch {epoch:2d}/{epochs}{' (head only)' if head_only else ''} | loss {log['loss']:.4f}"

        if sum_kd > 0:
            line += f" (gold {log['gold_loss']:.4f}, KD {log['kd_loss']:.4f})"

        line += (f" | train F1 {log['train_f1']:.4f} | dev F1 CL {dev_scores.get('CL', {}).get('f1', 0):.4f}"
                 f" IT {dev_scores.get('IT', {}).get('f1', 0):.4f} mean {dev_scores['score']:.4f}")

        stop = False

        if dev_scores["score"] > best_score:
            best_score, best_epoch, best_state = dev_scores["score"], epoch, trainable_state(model)
            line += "  <- best"
        elif epoch - best_epoch >= PATIENCE:
            stop = True

        if skipped:
            line += f" | skipped {skipped}"
        line += f" | {(time.time() - t_start) / 60:.1f} min"

        print(line, flush=True)
        history.append(log)

        if stop:
            print(f"Early stop: no improvement for {PATIENCE} epochs.")
            break

    model.load_state_dict(best_state, strict=False)          # back to the best epoch

    return history, best_epoch


def free_gpu_memory():

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

In [11]:
# ============================================================
# CELL 11 — STEP 1: TRAIN THE TEACHER
#   Architecture-1 decoder (Phase-1 weights FROZEN) + LoRMA + head,
#   trained on TRAIN, best epoch picked on dev
# ============================================================

def is_architecture1(path):

    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        return isinstance(ckpt, dict) and "rope_theta" in ckpt.get("model_config", {})
    except Exception:
        return False


if not TEACHER_CHECKPOINT.exists():

    found = [p for p in sorted(Path("/kaggle/input").rglob("best_model.pt")) if is_architecture1(p)]
    assert found, f"{TEACHER_CHECKPOINT} not found and no Architecture-1 checkpoint under /kaggle/input"

    TEACHER_CHECKPOINT = found[0]
    print("Teacher checkpoint found at:", TEACHER_CHECKPOINT)


phase1 = torch.load(TEACHER_CHECKPOINT, map_location="cpu", weights_only=False)
TEACHER_CONFIG = phase1["model_config"]

# decoder weights without the LM head (tied to the embedding, not needed here)
TEACHER_WEIGHTS = {
    key.replace("_orig_mod.", ""): value
    for key, value in phase1["model_state_dict"].items()
    if not key.replace("_orig_mod.", "").startswith("lm_head.")
}

del phase1

print("Teacher config:", TEACHER_CONFIG)

assert TEACHER_CONFIG["vocab_size"] == VOCAB_SIZE, "teacher was trained with a different tokenizer"
assert WINDOW <= TEACHER_CONFIG["block_size"], f"WINDOW must be <= {TEACHER_CONFIG['block_size']}"


def build_teacher():
    """Frozen Phase-1 decoder + LoRMA + new head."""

    backbone = DecoderBackbone(TEACHER_CONFIG)
    backbone.load_state_dict(TEACHER_WEIGHTS, strict=True)          # every name and shape must match

    add_lorma(backbone)

    return SentenceRoleModel(backbone, TEACHER_CONFIG["d_model"]).to(DEVICE)


# free GPU memory if this cell is re-run
for name in ["teacher", "student"]:
    globals().pop(name, None)

sys.last_traceback = sys.last_value = sys.last_type = None
free_gpu_memory()

teacher = build_teacher()

TEACHER_PARAMS = sum(p.numel() for p in teacher.parameters())
TEACHER_TRAINABLE = sum(p.numel() for p in teacher.parameters() if p.requires_grad)

print(f"Teacher: {TEACHER_PARAMS:,} parameters, {TEACHER_TRAINABLE:,} trainable (LoRMA + head)")


print(f"\nTEACHER | train {len(train_docs)} docs -> dev {len(dev_docs)} docs")

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

t0 = time.time()
teacher_history, teacher_best_epoch = train_model(
    teacher, teacher_loss, TEACHER_EPOCHS, TEACHER_WINDOWS_PER_CHUNK, LR_LORMA, 0.0
)
teacher_minutes = (time.time() - t0) / 60
teacher_gpu_gb = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0

torch.save(trainable_state(teacher), OUTPUT_DIR / "teacher_best_trainable.pt")

print(f"\n✅ Teacher trained in {teacher_minutes:.1f} min | best epoch {teacher_best_epoch} | peak GPU {teacher_gpu_gb:.2f} GB")

Teacher checkpoint found at: /kaggle/input/notebooks/iitianritik/decoder-only-archi-1/legal_lm/best_model.pt
Teacher config: {'vocab_size': 32004, 'block_size': 2048, 'd_model': 768, 'n_layers': 12, 'n_heads': 12, 'd_ff': 2048, 'rope_theta': 10000.0}
Teacher: 114,695,632 parameters, 5,161,168 trainable (LoRMA + head)

TEACHER | train 80 docs -> dev 10 docs
Epoch  1/15 (head only) | loss 2.8960 | train F1 0.3098 | dev F1 CL 0.2236 IT 0.2886 mean 0.2561  <- best | 2.2 min
Epoch  2/15 | loss 2.4217 | train F1 0.3116 | dev F1 CL 0.2744 IT 0.2865 mean 0.2805  <- best | skipped 2 | 4.4 min
Epoch  3/15 | loss 1.8219 | train F1 0.4447 | dev F1 CL 0.4354 IT 0.4106 mean 0.4230  <- best | 6.7 min
Epoch  4/15 | loss 1.4733 | train F1 0.5128 | dev F1 CL 0.3878 IT 0.4837 mean 0.4358  <- best | 8.9 min
Epoch  5/15 | loss 1.2156 | train F1 0.5890 | dev F1 CL 0.3655 IT 0.5082 mean 0.4368  <- best | skipped 1 | 11.1 min
Epoch  6/15 | loss 0.9368 | train F1 0.6297 | dev F1 CL 0.4194 IT 0.4704 mean 0.4449

In [12]:
# ============================================================
# CELL 12 — STEP 2: TEACHER SCORES FOR EVERY TRAIN SENTENCE
#   (raw scores; the temperature is applied in the KD loss)
#   + teacher on dev (for the comparison), then free its memory
# ============================================================

teacher.eval()

t0 = time.time()
teacher_correct = teacher_total = 0

for doc in train_docs:

    if not doc["sentences"]:
        doc["teacher_scores"] = torch.zeros(0, NUM_LABELS)
        continue

    doc["teacher_scores"] = torch.from_numpy(document_emissions(teacher, doc))

    teacher_correct += int((doc["teacher_scores"].argmax(dim=1) == torch.tensor(doc["label_ids"])).sum())
    teacher_total += len(doc["sentences"])

print(f"Teacher scores for {teacher_total} train sentences in {time.time() - t0:.0f}s | "
      f"teacher correct on {teacher_correct / max(teacher_total, 1):.1%} of them (KD uses only these)")

t0 = time.time()
teacher_dev, teacher_dev_predictions = evaluate(teacher, dev_docs)
teacher_seconds = time.time() - t0

print(f"Teacher dev macro-F1: CL {teacher_dev.get('CL', {}).get('f1', 0):.4f} | "
      f"IT {teacher_dev.get('IT', {}).get('f1', 0):.4f} | mean {teacher_dev['score']:.4f} | predict {teacher_seconds:.1f}s")

with open(OUTPUT_DIR / "teacher_dev_prediction.json", "w", encoding="utf-8") as f:
    json.dump(teacher_dev_predictions, f, ensure_ascii=False, indent=1)

# the student only needs the saved scores
teacher = None
free_gpu_memory()

Teacher scores for 16845 train sentences in 40s | teacher correct on 82.3% of them (KD uses only these)
Teacher dev macro-F1: CL 0.4222 | IT 0.5411 | mean 0.4816 | predict 5.1s


In [13]:
# ============================================================
# CELL 13 — STUDENT: Architecture-2 encoder (all weights trained) + head
# ============================================================

if not STUDENT_CHECKPOINT.exists():

    found = []
    for path in sorted(Path("/kaggle/input").rglob("best_encoder_decoder.pt")):
        try:
            if "encoder_layers" in torch.load(path, map_location="cpu", weights_only=False).get("config", {}):
                found.append(path)
        except Exception:
            pass

    assert found, f"{STUDENT_CHECKPOINT} not found and no Architecture-2 checkpoint under /kaggle/input"

    STUDENT_CHECKPOINT = found[0]
    print("Student checkpoint found at:", STUDENT_CHECKPOINT)


phase1 = torch.load(STUDENT_CHECKPOINT, map_location="cpu", weights_only=False)
STUDENT_CONFIG = dict(phase1["config"])

# encoder weights only (the decoder of Architecture-2 is not needed)
STUDENT_WEIGHTS = {}

for key, value in phase1["model_state_dict"].items():
    key = key.replace("_orig_mod.", "")
    if key.startswith("encoder."):
        STUDENT_WEIGHTS[key[len("encoder."):]] = value

del phase1

print("Student config:", {k: STUDENT_CONFIG[k] for k in ("d_model", "encoder_layers", "n_heads", "ffn_dim", "src_len")})

assert STUDENT_CONFIG["vocab_size"] == VOCAB_SIZE, "student was trained with a different tokenizer"
assert WINDOW <= STUDENT_CONFIG["src_len"], f"WINDOW must be <= {STUDENT_CONFIG['src_len']} (learned positions)"


def build_student():
    """Phase-1 Architecture-2 encoder + new head."""

    backbone = Arch2Encoder(STUDENT_CONFIG)
    backbone.load_state_dict(STUDENT_WEIGHTS, strict=True)          # every name and shape must match

    return SentenceRoleModel(backbone, STUDENT_CONFIG["d_model"]).to(DEVICE)


student = build_student()

STUDENT_PARAMS = sum(p.numel() for p in student.parameters())
STUDENT_TRAINABLE = sum(p.numel() for p in student.parameters() if p.requires_grad)

print(f"\nStudent: {STUDENT_PARAMS:,} parameters, all trainable "
      f"-> {STUDENT_PARAMS / TEACHER_PARAMS:.0%} of the teacher's size")

Student checkpoint found at: /kaggle/input/notebooks/ritikguptatab/proj-cap-p-1-archi-2/architecture_2_encoder_decoder/best_encoder_decoder.pt
Student config: {'d_model': 768, 'encoder_layers': 6, 'n_heads': 12, 'ffn_dim': 3072, 'src_len': 1024}

Student: 70,008,016 parameters, all trainable -> 61% of the teacher's size


In [14]:
# ============================================================
# CELL 14 — STEP 3: TRAIN THE STUDENT WITH DISTILLATION (TRAIN data only)
# ============================================================

print(f"STUDENT | {RUN_NAME} | train {len(train_docs)} docs -> dev {len(dev_docs)} docs | "
      f"KD alpha {KD_ALPHA}, T {KD_TEMPERATURE}")

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

t0 = time.time()
student_history, student_best_epoch = train_model(
    student, student_loss, STUDENT_EPOCHS, STUDENT_WINDOWS_PER_CHUNK, LR_STUDENT_ENCODER, WEIGHT_DECAY
)
student_minutes = (time.time() - t0) / 60
student_gpu_gb = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0

torch.save(trainable_state(student), OUTPUT_DIR / "student_best.pt")

print(f"\n✅ Student trained in {student_minutes:.1f} min | best epoch {student_best_epoch} | peak GPU {student_gpu_gb:.2f} GB")

STUDENT | p3_kd_arch1lorma_to_arch2 | train 80 docs -> dev 10 docs | KD alpha 0.5, T 2.0
Epoch  1/15 (head only) | loss 2.9720 (gold 3.0274, KD 2.9166) | train F1 0.1827 | dev F1 CL 0.1815 IT 0.1287 mean 0.1551  <- best | 1.1 min
Epoch  2/15 | loss 2.3541 (gold 2.4705, KD 2.2377) | train F1 0.2522 | dev F1 CL 0.3035 IT 0.2770 mean 0.2902  <- best | 2.1 min
Epoch  3/15 | loss 1.7407 (gold 1.8598, KD 1.6216) | train F1 0.3036 | dev F1 CL 0.2535 IT 0.2483 mean 0.2509 | 3.2 min
Epoch  4/15 | loss 1.4237 (gold 1.5527, KD 1.2948) | train F1 0.3576 | dev F1 CL 0.2900 IT 0.2265 mean 0.2583 | 4.3 min
Epoch  5/15 | loss 1.1245 (gold 1.2635, KD 0.9854) | train F1 0.5609 | dev F1 CL 0.3495 IT 0.3708 mean 0.3602  <- best | 5.3 min
Epoch  6/15 | loss 0.9314 (gold 1.0559, KD 0.8068) | train F1 0.5914 | dev F1 CL 0.3221 IT 0.3316 mean 0.3269 | 6.4 min
Epoch  7/15 | loss 0.7462 (gold 0.8522, KD 0.6402) | train F1 0.6427 | dev F1 CL 0.3196 IT 0.3407 mean 0.3302 | 7.5 min
Epoch  8/15 | loss 0.6447 (gold 

In [15]:
# ============================================================
# CELL 15 — DEV REPORT: teacher vs student, per label, confusions + save results
# ============================================================

t0 = time.time()
dev_scores, dev_predictions = evaluate(student, dev_docs)
student_seconds = time.time() - t0


# ------------------------------------------------------------
# Teacher vs student
# ------------------------------------------------------------

print(f"{'model':36s} {'params':>12s} {'trained':>12s} {'CL F1':>7s} {'IT F1':>7s} {'mean':>7s} {'train min':>9s} {'predict s':>9s}")

rows = [
    ("Teacher: Arch-1 + LoRMA", TEACHER_PARAMS, TEACHER_TRAINABLE, teacher_dev, teacher_minutes, teacher_seconds),
    (f"Student: Arch-2 encoder{' + KD' if KD_ALPHA > 0 else ' (no KD)'}", STUDENT_PARAMS, STUDENT_TRAINABLE,
     dev_scores, student_minutes, student_seconds),
]

for name, total, trained_count, scores, minutes, seconds in rows:
    print(f"{name:36s} {total:12,d} {trained_count:12,d} {scores.get('CL', {}).get('f1', 0):7.4f} "
          f"{scores.get('IT', {}).get('f1', 0):7.4f} {scores['score']:7.4f} {minutes:9.1f} {seconds:9.1f}")

print(f"\nStudent dev F1 = {dev_scores['score'] / max(teacher_dev['score'], 1e-9):.0%} of the teacher's, "
      f"with {STUDENT_PARAMS / TEACHER_PARAMS:.0%} of its parameters "
      f"and {student_seconds / max(teacher_seconds, 1e-9):.0%} of its prediction time.")

print(f"\nSTUDENT DEV ({RUN_NAME}, best epoch {student_best_epoch})")
print(f"{'':6s} {'macro-P':>8s} {'macro-R':>8s} {'macro-F1':>9s}")

for key in ["CL", "IT", "all"]:
    if key in dev_scores:
        s = dev_scores[key]
        print(f"{key:6s} {s['precision']:8.4f} {s['recall']:8.4f} {s['f1']:9.4f}")


# ------------------------------------------------------------
# Per label + most frequent mistakes
# ------------------------------------------------------------

per_label = {}

for domain in ["CL", "IT"]:

    gold, pred = flat_labels(dev_docs, dev_predictions, domain)

    if not gold:
        continue

    p, r, f, support = precision_recall_fscore_support(gold, pred, labels=LABELS, zero_division=0)

    per_label[domain] = {
        label: {"precision": float(p[k]), "recall": float(r[k]), "f1": float(f[k]), "support": int(support[k])}
        for k, label in enumerate(LABELS)
    }

    print(f"\n{domain}: {'label':24s} {'P':>6s} {'R':>6s} {'F1':>6s} {'gold':>5s}")

    for label in LABELS:
        s = per_label[domain][label]
        print(f"    {label:24s} {s['precision']:6.3f} {s['recall']:6.3f} {s['f1']:6.3f} {s['support']:5d}")

    confusions = Counter((g, q) for g, q in zip(gold, pred) if g != q)

    print(f"\n  most frequent mistakes in {domain} (gold -> predicted):")

    for (g, q), count in confusions.most_common(8):
        print(f"    {g:24s} -> {q:24s} {count:4d}")


# ------------------------------------------------------------
# Closely related roles
# ------------------------------------------------------------

RELATED_ROLES = {
    "precedents": ["PrecedentReliedUpon", "PrecedentNotReliedUpon", "PrecedentOverruled"],
    "arguments": ["ArgumentPetitioner", "ArgumentRespondent"],
    "decision": ["RatioOfTheDecision", "RulingByPresentCourt"],
    "background": ["Fact", "RulingByLowerCourt"],
}

print(f"\n{'group':12s} {'domain':6s} {'sentences':>9s} {'correct':>8s} {'confused within group':>22s}")

confusion_groups = {}

for group, roles in RELATED_ROLES.items():
    for domain in ["CL", "IT"]:

        gold, pred = flat_labels(dev_docs, dev_predictions, domain)
        pairs = [(g, q) for g, q in zip(gold, pred) if g in roles]

        if not pairs:
            continue

        correct = sum(g == q for g, q in pairs) / len(pairs)
        within = sum(g != q and q in roles for g, q in pairs) / len(pairs)

        confusion_groups[f"{group}_{domain}"] = {"sentences": len(pairs), "correct": correct, "within_group": within}

        print(f"{group:12s} {domain:6s} {len(pairs):9d} {correct:8.1%} {within:22.1%}")


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

results = {
    "run": RUN_NAME,
    "trained_on": "train only (dev used to pick the best epoch)",
    "teacher": {"model": "Architecture-1 decoder (frozen) + LoRMA",
                "lorma": {"rank": LORMA_RANK, "alpha": LORMA_ALPHA, "targets": LORMA_TARGETS},
                "parameters": TEACHER_PARAMS, "parameters_trainable": TEACHER_TRAINABLE,
                "best_epoch": teacher_best_epoch, "train_minutes": teacher_minutes, "peak_gpu_gb": teacher_gpu_gb,
                "dev": teacher_dev, "dev_predict_seconds": teacher_seconds, "history": teacher_history},
    "student": {"model": "Architecture-2 encoder (full fine-tuning)",
                "parameters": STUDENT_PARAMS, "parameters_trainable": STUDENT_TRAINABLE,
                "best_epoch": student_best_epoch, "train_minutes": student_minutes, "peak_gpu_gb": student_gpu_gb,
                "dev": dev_scores, "dev_per_label": per_label, "dev_related_roles": confusion_groups,
                "dev_predict_seconds": student_seconds, "history": student_history},
    "kd": {"alpha": KD_ALPHA, "temperature": KD_TEMPERATURE, "only_where_teacher_correct": KD_ONLY_WHERE_TEACHER_CORRECT},
    "never_predicted": {d: [l for l, ok in zip(LABELS, LABEL_ALLOWED[d]) if not ok] for d in ["CL", "IT"]},
}

with open(OUTPUT_DIR / "results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

with open(OUTPUT_DIR / "dev_prediction.json", "w", encoding="utf-8") as f:
    json.dump(dev_predictions, f, ensure_ascii=False, indent=1)

print("\nSaved:", OUTPUT_DIR / "results.json", "and dev_prediction.json")

model                                      params      trained   CL F1   IT F1    mean train min predict s
Teacher: Arch-1 + LoRMA               114,695,632    5,161,168  0.4222  0.5411  0.4816      26.8       5.1
Student: Arch-2 encoder + KD           70,008,016   70,008,016  0.3495  0.3708  0.3602      10.7       1.6


STUDENT DEV (p3_kd_arch1lorma_to_arch2, best epoch 5)
        macro-P  macro-R  macro-F1
CL       0.4285   0.3711    0.3495
IT       0.5334   0.3607    0.3708
all      0.5518   0.3754    0.3920

CL: label                         P      R     F1  gold
    Fact                      0.811  0.518  0.632   249
    Issue                     0.000  0.000  0.000     3
    ArgumentPetitioner        0.539  0.401  0.460   344
    ArgumentRespondent        0.663  0.233  0.345   236
    Statute                   0.680  0.879  0.767    58
    Dissent                   0.000  0.000  0.000     0
    PrecedentReliedUpon       0.208  0.246  0.226   142
    PrecedentNotReliedUpon    0.00

In [16]:
# ============================================================
# CELL 16 — PREDICT TEST WITH THE STUDENT + WRITE prediction.json + submission.zip
# ============================================================

test_predictions = predict_labels(student, test_docs)


# ------------------------------------------------------------
# Format checks: one valid label per sentence, same order, every document
# ------------------------------------------------------------

ids = [doc["id"] for doc in test_docs]
assert len(ids) == len(set(ids)), "the same document id appears in CL and IT test files"

for doc in test_docs:

    labels = test_predictions[doc["id"]]

    assert len(labels) == len(doc["sentences"]), doc["id"]
    assert all(label in LABEL2ID for label in labels), doc["id"]

print(f"✅ {len(test_docs)} test documents, {sum(len(v) for v in test_predictions.values())} sentences, all checks passed")

print("\nPredicted label distribution (test):")
for label, count in Counter(l for v in test_predictions.values() for l in v).most_common():
    print(f"  {label:24s} {count:6d}")


PREDICTION_FILE = OUTPUT_DIR / "prediction.json"
ZIP_FILE = OUTPUT_DIR / "submission.zip"

with open(PREDICTION_FILE, "w", encoding="utf-8") as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=1)

# the same predictions split by domain (in case Codabench wants one file per domain)
for domain in ["CL", "IT"]:
    with open(OUTPUT_DIR / f"{domain}_prediction.json", "w", encoding="utf-8") as f:
        json.dump({d["id"]: test_predictions[d["id"]] for d in test_docs if d["domain"] == domain},
                  f, ensure_ascii=False, indent=1)

with zipfile.ZipFile(ZIP_FILE, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(PREDICTION_FILE, arcname="prediction.json")        # file at the ROOT of the zip

with zipfile.ZipFile(ZIP_FILE) as zf:
    print("\nsubmission.zip:", zf.namelist())

print("Saved:", ZIP_FILE)

✅ 10 test documents, 2197 sentences, all checks passed

Predicted label distribution (test):
  PrecedentReliedUpon         583
  RatioOfTheDecision          582
  Fact                        352
  ArgumentPetitioner          308
  Statute                     179
  PrecedentNotReliedUpon       69
  ArgumentRespondent           46
  RulingByPresentCourt         37
  RulingByLowerCourt           35
  None                          6

submission.zip: ['prediction.json']
Saved: /kaggle/working/p3_kd_arch1lorma_to_arch2/submission.zip
